## Setup

In [1]:
%load_ext autoreload
%autoreload 2

from pyo.lib.server import Server

from pysynth.clock import Clock
from pysynth.tempo import Tempo
from pysynth.patches import PatchRack, atmosphere, bass, clock_tick, drone, hat, low_hat

s = Server().boot()
s.start()

tempo = Tempo(bpm=132)
rack = PatchRack()

# single shared pulse every patch (except clock_tick, which is deliberately
# free-running) divides down from, so they lock to a common downbeat instead
# of just a common tempo - see src/pysynth/clock.py
clock = Clock(tempo)
clock.start()

Pyo warning: Portaudio input device `MacBook Pro Microphone` has fewer channels (1) than requested (2).
Pyo warning: Portmidi warning: no midi device found!
Portmidi closed.


## Bass

Rolling 16-step bassline through a resonant, LFO-swept lowpass filter. See `src/pysynth/patches/bass.py`.

In [2]:
from ipywidgets import FloatSlider, interact


def set_bass_params(root_freq, filter_res, filter_base, filter_range, volume):
    bass_params = {
        "root_freq": root_freq,
        "filter_res": filter_res,
        "filter_base": filter_base,
        "filter_range": filter_range,
        "volume": volume,
    }
    bass_patch = rack.toggle("bass", bass.build, tempo, clock, **bass_params)


interact(
    set_bass_params,
    root_freq=FloatSlider(min=30, max=110, step=1, value=bass.ROOT_FREQ, description="root_freq"),
    filter_res=FloatSlider(min=0, max=1, step=0.05, value=0.75, description="filter_res"),
    filter_base=FloatSlider(min=200, max=2000, step=10, value=900, description="filter_base"),
    filter_range=FloatSlider(min=0, max=1000, step=10, value=400, description="filter_range"),
    volume=FloatSlider(min=0, max=2, step=0.1, value=1.0, description="volume"),
)

interactive(children=(FloatSlider(value=55.0, description='root_freq', max=110.0, min=30.0, step=1.0), FloatSl…

<function __main__.set_bass_params(root_freq, filter_res, filter_base, filter_range, volume)>

In [ ]:
bass_params = {
    # Fundamental frequency of the bassline's root note.
    "root_freq": bass.ROOT_FREQ,  # default 55, range 30-110
    # Filter resonance - higher gives a more squelchy, whistling character.
    "filter_res": 0.9,  # default 0.75, range 0-1
    # Center cutoff the sweep rides on - higher is brighter.
    "filter_base": 900,  # default 900, range 200-2000
    # How far the cutoff sweeps each bar - larger is a more dramatic sweep.
    "filter_range": 400,  # default 400, range 0-1000
    # Output level for this patch (limited via Compress so it won't clip).
    "volume": 0.7,  # default 1.0, range 0-2
}
bass_patch = rack.toggle("bass", bass.build, tempo, clock, **bass_params)

In [ ]:
bass_patch.